## **LangChain** - _Embeddings_

> Un **Embedding** es una técnica que traduce lenguaje humano (palabras, frases o documentos enteros) en secuencias de números (vectores de alta dimensionalidad) que las máquinas pueden procesar. Su mayor cualidad es que capturan el significado semántico: textos con conceptos similares generarán vectores que matemáticamente apuntan en la misma dirección o están muy cerca en un espacio multidimensional.

### Características principales:
*   **Densidad:** A diferencia de métodos clásicos (como Bag of Words), los vectores son densos (pocos ceros).
*   **Espacio Vectorial:** Los embeddings sitúan las palabras en un espacio multidimensional. Palabras con significados similares terminan en coordenadas cercanas dentro de ese espacio.
*   **Relaciones Matemáticas:** Permiten realizar operaciones matemáticas. El ejemplo clásico es:
    $$ Rey - Hombre + Mujer \approx Reina $$

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.utils.math import cosine_similarity
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from dotenv import load_dotenv

load_dotenv()

True

### 01. Embeddings

In [21]:
# embeddings_model = HuggingFaceEmbeddings(model_name = "paraphrase-multilingual-MiniLM-L12-v2") # El tamaño del embedding depende del modelo
embeddings_model = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-2", output_dimensionality = 3072)

texto = "Probando embeddings locales"

# Generar el embedding
embedding = embeddings_model.embed_query(text = texto)

print(f"Dimensión del vector: {len(embedding)}")

Dimensión del vector: 3072


In [22]:
embedding

[-0.040129438,
 0.010985082,
 -0.0051997826,
 0.0077607557,
 0.011605473,
 0.021876795,
 -0.029950231,
 -0.0005221574,
 0.008723768,
 -0.028013866,
 -0.02588593,
 -0.02265294,
 -0.0029961031,
 -0.010288083,
 -0.00075310655,
 0.0035042865,
 0.041808818,
 -0.014778714,
 -0.01256899,
 0.012218627,
 0.012691155,
 0.010661625,
 0.003992447,
 0.009468582,
 -0.011320367,
 0.014028386,
 0.008787166,
 -0.002142638,
 -0.022492027,
 0.13116969,
 -0.022225192,
 -0.0218638,
 -0.028728956,
 -0.0023726937,
 0.013426936,
 -0.016188264,
 -0.00089745666,
 0.0074690823,
 0.015561627,
 -0.021350943,
 0.019585203,
 0.029574204,
 0.020286588,
 0.004859243,
 0.028992,
 0.010486681,
 -0.0076684174,
 0.0005562447,
 -0.019728918,
 -0.017618684,
 -0.03040582,
 -0.014464684,
 -0.005807844,
 -0.016279701,
 -0.0010424831,
 -0.0048338897,
 0.007832323,
 -0.0019239924,
 0.02115312,
 0.017330142,
 0.011958654,
 -0.022033604,
 0.013386441,
 0.009517005,
 0.0049791033,
 0.012533009,
 0.009326368,
 -0.0073271277,
 0.0173

In [23]:
oraciones = ["Me encanta programar en Python.",
             "El desarrollo de software con Python es genial.",
             "Hoy hace un día soleado en Madrid.",
             "El clima está despejado y brillante hoy.",
             "El papa está en Barcelona."]

embeddings = embeddings_model.embed_documents(texts = oraciones)

print(f"Documentos procesados: {len(embeddings)}")
print(f"Dimensión del primer documento: {len(embeddings[0])}")

Documentos procesados: 5
Dimensión del primer documento: 3072


### 02. Similitud Semántica

La similitud semántica es una métrica empleada en lingüística y en inteligencia artificial que mide el grado en que dos palabras, frases o conceptos comparten un mismo significado o intención, independientemente de las palabras literales que utilicen.

A diferencia de la coincidencia exacta de texto (que busca palabras iguales), la similitud semántica comprende el contexto. Permite a las máquinas entender que frases diferentes tienen un propósito equivalente.

- Para calcular la similitud semántica se utiliza principalmente la similitud de coseno:
    - Esta función mide el coseno del ángulo formado por dos vectores en un espacio multidimensional. Cuando los textos se transforman en vectores, esta fórmula evalúa qué tan alineados están sus significados.
    
$$
\text{Similitud de Coseno} = \frac{\mathbf{A}\cdot \mathbf{B}}{\|\mathbf{A}\|\|\mathbf{B}\|}=\frac{\sum _{i=1}^{n}A_{i}B_{i}}{\sqrt{\sum _{i=1}^{n}A_{i}^{2}}\sqrt{\sum _{i=1}^{n}B_{i}^{2}}}\
$$


- La razón fundamental por la que se elige esta función sobre otras (como la distancia Euclidiana) es que ignora la longitud de los textos y se enfoca estrictamente en la dirección del significado:

    - **Independencia del tamaño**: Si un documento repite la palabra "computadora" 10 veces y otro texto corto la menciona solo una vez, sus vectores apuntarán en la misma dirección, pero uno será mucho más largo que el otro. El coseno ignora esa diferencia de tamaño.
    - **Escala normalizada**: El resultado siempre oscila en un rango fijo entre -1 y 1 (o entre 0 y 1) en la mayoría de los sistemas de embeddings. Esto facilita establecer umbrales automáticos (por ejemplo, decidir que cualquier resultado mayor a 0.85 es un sinónimo).
    - **Interpretación del resultados:**
        - Resultado igual a 1: Los vectores son idénticos y apuntan exactamente en la misma dirección. Significa máxima similitud semántica.
        - Resultado igual a 0: Los vectores son perpendiculares u ortogonales. Significa que no comparten ninguna relación semántica.
        - Resultado igual a -1: Los vectores apuntan en direcciones opuestas. Significa que son conceptos opuestos.

In [24]:
# Calculamos la similitud del coseno entre todas las oraciones
similitudes = cosine_similarity(X = embeddings, Y = embeddings)

pd.DataFrame(similitudes)

,0,1,2,3,4
0,1.000000,0.751021,0.458424,0.401609,0.367353
1,0.751021,1.000000,0.424975,0.436193,0.356319
2,0.458424,0.424975,1.000000,0.705500,0.543731
3,0.401609,0.436193,0.705500,1.000000,0.365269
4,0.367353,0.356319,0.543731,0.365269,1.000000


In [25]:
oraciones

['Me encanta programar en Python.',
 'El desarrollo de software con Python es genial.',
 'Hoy hace un día soleado en Madrid.',
 'El clima está despejado y brillante hoy.',
 'El papa está en Barcelona.']

In [26]:
# Oración 0 y la oración 1
print(f"Oración 1: '{oraciones[0]}'")
print(f"Oración 2: '{oraciones[1]}'")
print(f"Puntuación de similitud: {similitudes[0][1]:.4f}")

# Oración 0 y la oración 2
print(f"\nOración 1: '{oraciones[0]}'")
print(f"Oración 2: '{oraciones[2]}'")
print(f"Puntuación de similitud: {similitudes[0][2]:.4f}")

Oración 1: 'Me encanta programar en Python.'
Oración 2: 'El desarrollo de software con Python es genial.'
Puntuación de similitud: 0.7510

Oración 1: 'Me encanta programar en Python.'
Oración 2: 'Hoy hace un día soleado en Madrid.'
Puntuación de similitud: 0.4584


### 03. Búsqueda Semántica (Motor de Búsqueda de IA)

Buscar información por su significado y no por palabras clave exactas.

In [35]:
corpus = [
    "Un perro está persiguiendo una pelota.",
    "Un hombre está comiendo una manzana.",
    "La inteligencia artificial está transformando el mundo.",
    "Un cachorro juega con su juguete en el parque."
]

consulta = "Un animal pequeño divirtiéndose"

# Creamos el VectorStore directamente con los textos y el modelo
vectorstore = FAISS.from_texts(texts = corpus,
                               embedding = embeddings_model,
                               distance_strategy = DistanceStrategy.COSINE)

# Realizamos la búsqueda semántica. 
resultados = vectorstore.similarity_search_with_score(query = consulta, k = 4)

print(f"Consulta: {consulta}\n")
for doc, score in resultados:
    print(f"Documento encontrado: '{doc.page_content}' (Similitud: {score:.4f})")

Consulta: Un animal pequeño divirtiéndose

Documento encontrado: 'Un perro está persiguiendo una pelota.' (Similitud: 0.8565)
Documento encontrado: 'Un cachorro juega con su juguete en el parque.' (Similitud: 0.9509)
Documento encontrado: 'Un hombre está comiendo una manzana.' (Similitud: 1.0323)
Documento encontrado: 'La inteligencia artificial está transformando el mundo.' (Similitud: 1.0900)


In [34]:
resultados[2]

(Document(id='7f7d6d9b-e729-4f23-99d8-b0a835728bbf', metadata={}, page_content='Un hombre está comiendo una manzana.'),
 np.float32(1.03232))

### 04. Clustering (Agrupación de Textos)

Podemos usar embeddings junto con algoritmos tradicionales de Machine Learning para agrupar textos similares automáticamente, ideal para organizar tickets de soporte o reseñas de clientes.

In [28]:
textos_variados = ["La nueva tarjeta gráfica es rapidísima.",
                   "El pastel de chocolate estaba delicioso.",
                   "Mi monitor parpadea y no enciende.",
                   "Necesito la receta de la tarta de queso.",
                   "El procesador se calienta demasiado.",
                   "Es el momento de pedir una tarta red velvet.",
                   "Las tarjetas graficas están muy costosas.",
                   "El otro día desayuné un monitor."]

embeddings = embeddings_model.embed_documents(texts = textos_variados)

# Queremos agruparlos en 2 categorías (Tecnología vs Comida)
num_clusters = 2
kmeans = KMeans(n_clusters = num_clusters, random_state = 42)
kmeans.fit(embeddings)

grupos = kmeans.labels_

print("Agrupación automática de textos:")

for i in range(num_clusters):

    print(f"--- Clúster {i + 1} ---")

    for texto, grupo in zip(textos_variados, grupos):

        if grupo == i:

            print(f"- {texto}")

Agrupación automática de textos:
--- Clúster 1 ---
- La nueva tarjeta gráfica es rapidísima.
- Mi monitor parpadea y no enciende.
- El procesador se calienta demasiado.
- Las tarjetas graficas están muy costosas.
- El otro día desayuné un monitor.
--- Clúster 2 ---
- El pastel de chocolate estaba delicioso.
- Necesito la receta de la tarta de queso.
- Es el momento de pedir una tarta red velvet.


### 05. Visualización 2D de Embeddings

In [29]:
# Reducción de Dimensionalidad
pca = PCA(n_components = 2)
embeddings_2d = pca.fit_transform(embeddings)

df = pd.DataFrame(data = embeddings_2d, columns = ["pca1", "pca2"])
df["grupos"] = grupos
df["texto"] = textos_variados

# Scatter Plot
px.scatter(data_frame = df, x = "pca1", y = "pca2", color = "grupos", hover_name = "texto")